In [5]:
from datasets import load_from_disk
import numpy as np
from tqdm import tqdm
ds_train = load_from_disk("../../../datasets/coco_train_features_resnet_152")
ds_val = load_from_disk("../../../datasets/coco_val_features_resnet_152")
split = ds_val.train_test_split(test_size=1500, seed=42)
ds_val, ds_test = split['test'], split['train']

ds_train, ds_val, ds_test

(Dataset({
     features: ['coco_url', 'captions', 'image_id', 'features'],
     num_rows: 118287
 }),
 Dataset({
     features: ['coco_url', 'captions', 'image_id', 'features'],
     num_rows: 1500
 }),
 Dataset({
     features: ['coco_url', 'captions', 'image_id', 'features'],
     num_rows: 3500
 }))

In [6]:
def create_exhaustive_tuples(features, n_distractors=3, shuffle=True, seed=42):
    """
    Each object is used once as target.
    Distractors are sampled randomly but reproducibly if seed is set.
    """
    num_objects = features.shape[0]
    rng = np.random.default_rng(seed)

    tuples = []
    labels = []

    all_indices = np.arange(num_objects)

    for i in tqdm(range(num_objects)):
        candidates = np.delete(all_indices, i)
        distractor_idxs = rng.choice(candidates, size=n_distractors, replace=False)

        target = features[i]
        distractors = features[distractor_idxs]

        tuple_vectors = np.vstack([target[None, :], distractors])

        if shuffle:
            perm = rng.permutation(n_distractors + 1)
            tuple_vectors = tuple_vectors[perm]
            label = int(np.where(perm == 0)[0][0])
        else:
            label = 0

        tuples.append(tuple_vectors)
        labels.append(label)

    return np.array(tuples), np.array(labels)

In [8]:
n_distractors = 5
train_tuples, train_labels = create_exhaustive_tuples(np.array(ds_train["features"]), n_distractors=n_distractors, shuffle=True, seed=42)
test_tuples, test_labels = create_exhaustive_tuples(np.array(ds_test["features"]), n_distractors=n_distractors, shuffle=True, seed=42)
valid_tuples, valid_labels = create_exhaustive_tuples(np.array(ds_val["features"]), n_distractors=n_distractors, shuffle=True, seed=42)

# valid_tuples = test_valid_tuples[:1500]
# valid_labels = test_valid_labels[:1500]
# test_tuples = test_valid_tuples[1500:]
# test_labels = test_valid_labels[1500:]

100%|██████████| 1500/1500 [00:00<00:00, 17533.40it/s]


In [ ]:
# train_tuples.shape

(4096, 4, 1000)

In [ ]:
# for t in train_tuples:
#     for item in t:
#         print(item[0])
#     print("----")

In [9]:
np.savez_compressed(
    f"../../../datasets/coco_npz/coco_full_train_val_152_resnet_{n_distractors}_distractors.npz",
    train=train_tuples,
    train_labels=train_labels,
    valid=valid_tuples,
    valid_labels=valid_labels,
    test=test_tuples,
    test_labels=test_labels,
    n_distractors=3
)